# pems12 按年节点增长数据处理

规则：
- 扫描 pems12 目录下 `d12_text_station_5min_YYYY_MM_DD.txt.gz`。
- **首年**：以数据集**首日**的 stations 作为该年的固定节点集合。
- **其余年份**：以每年 **1月1日** 当天的 stations 作为该年的节点集合。
- 每年只保留「该年锚定日」出现的节点，每年保存为一个 CSV。

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
from collections import defaultdict

In [2]:
PEMS_DIR = './pems12'
OUTPUT_DIR = '.'
PREFIX = 'd12_text_station_5min'
SUFFIX = '.txt.gz'

COLUMN_NAMES = [
    "Timestamp", "Station", "District", "Freeway #",
    "Direction of Travel", "Lane Type", "Station Length",
    "Samples", "% Observed", "Total Flow", "Avg Occupancy", "Avg Speed"
]

## 1. 读取 pems12 目录，解析所有可用日期

In [3]:
def list_pems_dates(data_dir):
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"目录不存在: {data_dir}")
    dates = {}
    for f in os.listdir(data_dir):
        if not f.startswith(PREFIX) or not f.endswith(SUFFIX):
            continue
        try:
            rest = f[len(PREFIX):-len(SUFFIX)].strip(' _')
            parts = rest.split('_')
            if len(parts) == 3:
                y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
                dt = date(y, m, d)
                dates[dt] = os.path.join(data_dir, f)
        except (ValueError, IndexError):
            continue
    return dates

available = list_pems_dates(PEMS_DIR)
sorted_dates = sorted(available.keys())
first_date = min(sorted_dates)
print(f"pems12 共 {len(available)} 个日期文件")
print(f"首日: {first_date}, 末日: {sorted_dates[-1]}")

pems12 共 8764 个日期文件
首日: 2002-01-01, 末日: 2025-12-31


## 2. 确定每年的锚定日与节点集合

In [4]:
def get_anchor_date(year, first_date):
    """首年用数据集首日，其余年份用该年 1 月 1 日。"""
    if year == first_date.year:
        return first_date
    return date(year, 1, 1)

def read_station_ids_from_file(path):
    df = pd.read_csv(path, header=None, usecols=[1], names=['Station'], compression='gzip')
    return np.sort(df['Station'].dropna().unique().astype(int))

years = sorted(set(d.year for d in sorted_dates))
year_anchor = {}
year_nodes = {}

for y in years:
    anchor = get_anchor_date(y, first_date)
    year_anchor[y] = anchor
    if anchor not in available:
        print(f"警告: {y} 年锚定日 {anchor} 无数据文件，跳过该年")
        continue
    path = available[anchor]
    year_nodes[y] = read_station_ids_from_file(path)
    print(f"{y} 年 锚定日={anchor}, 节点数={len(year_nodes[y])}")

2002 年 锚定日=2002-01-01, 节点数=1783
2003 年 锚定日=2003-01-01, 节点数=1783
2004 年 锚定日=2004-01-01, 节点数=1833
2005 年 锚定日=2005-01-01, 节点数=1833
2006 年 锚定日=2006-01-01, 节点数=1887
2007 年 锚定日=2007-01-01, 节点数=2050
2008 年 锚定日=2008-01-01, 节点数=2101
2009 年 锚定日=2009-01-01, 节点数=2229
2010 年 锚定日=2010-01-01, 节点数=2231
2011 年 锚定日=2011-01-01, 节点数=2318
2012 年 锚定日=2012-01-01, 节点数=2318
2013 年 锚定日=2013-01-01, 节点数=2336
2014 年 锚定日=2014-01-01, 节点数=2421
2015 年 锚定日=2015-01-01, 节点数=2365
2016 年 锚定日=2016-01-01, 节点数=2413
2017 年 锚定日=2017-01-01, 节点数=2442
2018 年 锚定日=2018-01-01, 节点数=2538
2019 年 锚定日=2019-01-01, 节点数=2539
2020 年 锚定日=2020-01-01, 节点数=2420
2021 年 锚定日=2021-01-01, 节点数=2481
2022 年 锚定日=2022-01-01, 节点数=2511
2023 年 锚定日=2023-01-01, 节点数=2511
2024 年 锚定日=2024-01-01, 节点数=2587
2025 年 锚定日=2025-01-01, 节点数=2587


## 3. 按年汇总：只保留锚定日节点，每年保存一个 CSV

In [5]:
def read_one_day(path, columns=None):
    if columns is None:
        columns = list(range(12))
    df = pd.read_csv(path, header=None, usecols=columns,
                    names=[COLUMN_NAMES[i] for i in columns], compression='gzip')
    return df


def generate_empty_day_df(day, node_set):
    """当某天文件缺失时，为该天所有节点生成 5 分钟分辨率、全 0 的占位数据。"""
    # PeMS 5min: 24*60/5 = 288 个时间点
    start_dt = datetime(day.year, day.month, day.day, 0, 0)
    timestamps = [
        (start_dt + timedelta(minutes=5 * i)).strftime("%m/%d/%Y %H:%M:%S")
        for i in range(288)
    ]

    rows = []
    for ts in timestamps:
        for st in node_set:
            rows.append([
                ts,          # Timestamp
                st,          # Station
                np.nan,      # District
                np.nan,      # Freeway #
                np.nan,      # Direction of Travel
                np.nan,      # Lane Type
                np.nan,      # Station Length
                0,           # Samples
                0,           # % Observed
                0,           # Total Flow
                0,           # Avg Occupancy
                0,           # Avg Speed
            ])

    return pd.DataFrame(rows, columns=COLUMN_NAMES)


os.makedirs(OUTPUT_DIR, exist_ok=True)

for year in years:
    if year not in year_nodes:
        continue
    node_set = set(year_nodes[year])
    start = first_date if year == first_date.year else date(year, 1, 1)
    end = date(year, 12, 31)
    chunks = []
    current = start
    while current <= end:
        if current in available:
            path = available[current]
            df = read_one_day(path)
            df = df[df['Station'].isin(node_set)]
        else:
            # 该天文件缺失，补 0
            df = generate_empty_day_df(current, node_set)
        chunks.append(df)
        current += timedelta(days=1)
    if not chunks:
        print(f"{year} 年无可用数据，跳过")
        continue
    out_df = pd.concat(chunks, ignore_index=True)
    out_path = os.path.join(OUTPUT_DIR, f'pems12_{year}.csv')
    out_df.to_csv(out_path, index=False)
    print(f"{year}: 节点数={len(node_set)}, 行数={len(out_df)}, 已保存 -> {out_path}")

2002: 节点数=1783, 行数=187386168, 已保存 -> .\pems12_2002.csv
2003: 节点数=1783, 行数=184399368, 已保存 -> .\pems12_2003.csv
2004: 节点数=1833, 行数=193168872, 已保存 -> .\pems12_2004.csv
2005: 节点数=1833, 行数=190801944, 已保存 -> .\pems12_2005.csv
2006: 节点数=1887, 行数=196789128, 已保存 -> .\pems12_2006.csv
2007: 节点数=2050, 行数=215354376, 已保存 -> .\pems12_2007.csv
2008: 节点数=2101, 行数=218952684, 已保存 -> .\pems12_2008.csv
2009: 节点数=2229, 行数=233347392, 已保存 -> .\pems12_2009.csv
2010: 节点数=2231, 行数=234286332, 已保存 -> .\pems12_2010.csv
2011: 节点数=2318, 行数=243612528, 已保存 -> .\pems12_2011.csv
2012: 节点数=2318, 行数=239869848, 已保存 -> .\pems12_2012.csv
2013: 节点数=2336, 行数=242368560, 已保存 -> .\pems12_2013.csv
2014: 节点数=2421, 行数=241052950, 已保存 -> .\pems12_2014.csv
2015: 节点数=2365, 行数=242223874, 已保存 -> .\pems12_2015.csv
2016: 节点数=2413, 行数=249622031, 已保存 -> .\pems12_2016.csv
2017: 节点数=2442, 行数=255967821, 已保存 -> .\pems12_2017.csv
2018: 节点数=2538, 行数=259209211, 已保存 -> .\pems12_2018.csv
2019: 节点数=2539, 行数=257519433, 已保存 -> .\pems12_2019.csv
2020: 节点数=

## 4. 各年节点数汇总

In [6]:
summary = pd.DataFrame({
    'year': list(year_nodes.keys()),
    'anchor_date': [year_anchor[y] for y in year_nodes],
    'n_nodes': [len(year_nodes[y]) for y in year_nodes],
}).sort_values('year').reset_index(drop=True)
summary

,year,anchor_date,n_nodes
0,2002,2002-01-01,1783
1,2003,2003-01-01,1783
2,2004,2004-01-01,1833
3,2005,2005-01-01,1833
4,2006,2006-01-01,1887
5,2007,2007-01-01,2050
6,2008,2008-01-01,2101
7,2009,2009-01-01,2229
8,2010,2010-01-01,2231
9,2011,2011-01-01,2318


## 5. Pivot 处理：行=5min时间步, 列=Station, 值=Total Flow

读取每年的 `pems12_{year}.csv`，pivot 成：
- **行**：完整的 5 分钟时间序列（缺失时间步自动补 NaN）
- **列**：每个 Station ID
- **值**：Total Flow

输出保存为 `pems12_{year}_flow.csv`

In [1]:
import os, re, glob
import pandas as pd
from datetime import date

FLOW_OUTPUT_DIR = '.'
FIRST_DATE = date(2002, 1, 1)
DISTRICT = 'pems12'

_csv_files = glob.glob(os.path.join(FLOW_OUTPUT_DIR, f'{DISTRICT}_[0-9][0-9][0-9][0-9].csv'))
years = sorted(int(re.search(rf'{DISTRICT}_(\d{{4}})\.csv', f).group(1)) for f in _csv_files)
print(f"扫描到 {len(years)} 个年度 CSV: {years[0]}~{years[-1]}")

def pivot_yearly_csv(year, first_date=FIRST_DATE, output_dir=FLOW_OUTPUT_DIR):
    src_path = os.path.join(output_dir, f'{DISTRICT}_{year}.csv')
    if not os.path.exists(src_path):
        print(f"  [跳过] {src_path} 不存在")
        return

    df = pd.read_csv(src_path, usecols=['Timestamp', 'Station', 'Total Flow'])
    print(f"  读取完成: {len(df):,} 行")

    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    pivot = df.pivot_table(
        index='Timestamp', columns='Station',
        values='Total Flow', aggfunc='sum'
    )
    del df

    pivot.columns = pivot.columns.astype(int)
    pivot = pivot[sorted(pivot.columns)]

    if year == first_date.year:
        start_dt = pd.Timestamp(first_date)
    else:
        start_dt = pd.Timestamp(year, 1, 1)
    end_dt = pd.Timestamp(year, 12, 31, 23, 55)

    full_idx = pd.date_range(start=start_dt, end=end_dt, freq='5min')
    pivot = pivot.reindex(full_idx)
    pivot.index.name = 'Timestamp'

    out_path = os.path.join(output_dir, f'{DISTRICT}_{year}_flow.csv')
    pivot.to_csv(out_path)

    n_nan_rows = pivot.isna().all(axis=1).sum()
    print(f"  {year}: stations={pivot.shape[1]}, "
          f"时间步={pivot.shape[0]}(期望{len(full_idx)}), "
          f"全NaN行={n_nan_rows}, 已保存 -> {out_path}")
    return pivot.shape

print("pivot_yearly_csv 函数已定义")

扫描到 24 个年度 CSV: 2002~2025
pivot_yearly_csv 函数已定义


In [2]:
import time

results = {}
for year in years:
    print(f"\n{'='*60}")
    print(f"处理 {year} 年 ...")
    t0 = time.time()
    shape = pivot_yearly_csv(year)
    elapsed = time.time() - t0
    print(f"  耗时: {elapsed:.1f}s")
    if shape:
        results[year] = shape

print(f"\n{'='*60}")
print(f"全部完成! 共处理 {len(results)} 年")


处理 2002 年 ...
  读取完成: 187,386,168 行
  2002: stations=1783, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems12_2002_flow.csv
  耗时: 1302.7s

处理 2003 年 ...
  读取完成: 184,399,368 行
  2003: stations=1783, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems12_2003_flow.csv
  耗时: 1388.6s

处理 2004 年 ...
  读取完成: 193,168,872 行
  2004: stations=1833, 时间步=105408(期望105408), 全NaN行=24, 已保存 -> .\pems12_2004_flow.csv
  耗时: 1324.4s

处理 2005 年 ...
  读取完成: 190,801,944 行
  2005: stations=1833, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems12_2005_flow.csv
  耗时: 1113.5s

处理 2006 年 ...
  读取完成: 196,789,128 行
  2006: stations=1887, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems12_2006_flow.csv
  耗时: 1132.7s

处理 2007 年 ...
  读取完成: 215,354,376 行
  2007: stations=2050, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems12_2007_flow.csv
  耗时: 1010.2s

处理 2008 年 ...
  读取完成: 218,952,684 行
  2008: stations=2101, 时间步=105408(期望105408), 全NaN行=24, 已保存 -> .\pems12_2008_flow.csv
  耗时: 1209.7s

处理 2009 年 ...
  读取完成: 233,347,392 行
  2009: sta

## 6. 检查输出结果汇总

In [3]:
summary_flow = pd.DataFrame([
    {'year': y, 'rows': s[0], 'stations': s[1]}
    for y, s in results.items()
])
summary_flow = summary_flow.sort_values('year').reset_index(drop=True)
print("各年 flow pivot 结果汇总:")
summary_flow

各年 flow pivot 结果汇总:


,year,rows,stations
0,2002,105120,1783
1,2003,105120,1783
2,2004,105408,1833
3,2005,105120,1833
4,2006,105120,1887
5,2007,105120,2050
6,2008,105408,2101
7,2009,105120,2229
8,2010,105120,2231
9,2011,105120,2318


In [4]:
check = pd.read_csv(f'./{DISTRICT}_{years[0]}_flow.csv', index_col=0, parse_dates=True, nrows=10)
print(f"{DISTRICT}_{years[0]}_flow.csv 前 10 行, shape={check.shape}")
check

pems12_2002_flow.csv 前 10 行, shape=(10, 1783)


,1201050,1201052,1201054,1201064,1201066,1201076,1201081,1201085,1201087,1201093,...,1212622,1212627,1212664,1212672,1212673,1212677,1212689,1212690,1212691,1212692
Timestamp,,,,,,,,,,,,,,,,,,,,,
2002-01-01 00:00:00,0.0,0.0,11.0,0.0,10.0,58.0,0.0,5.0,63.0,5.0,...,0.0,71.0,0.0,71.0,4.0,0.0,71.0,4.0,0.0,0.0
2002-01-01 00:05:00,0.0,0.0,11.0,0.0,9.0,50.0,0.0,4.0,46.0,14.0,...,0.0,63.0,0.0,63.0,4.0,0.0,63.0,4.0,0.0,0.0
2002-01-01 00:10:00,0.0,0.0,15.0,0.0,11.0,65.0,0.0,2.0,63.0,26.0,...,0.0,67.0,0.0,67.0,3.0,0.0,67.0,3.0,0.0,0.0
2002-01-01 00:15:00,0.0,0.0,12.0,0.0,10.0,55.0,1.0,0.0,82.0,27.0,...,0.0,75.0,0.0,75.0,4.0,0.0,75.0,4.0,0.0,0.0
2002-01-01 00:20:00,0.0,0.0,19.0,0.0,18.0,68.0,0.0,5.0,96.0,18.0,...,0.0,69.0,0.0,69.0,4.0,0.0,69.0,4.0,0.0,0.0
2002-01-01 00:25:00,0.0,0.0,26.0,0.0,22.0,80.0,0.0,8.0,125.0,27.0,...,0.0,66.0,0.0,66.0,4.0,0.0,66.0,4.0,0.0,0.0
2002-01-01 00:30:00,0.0,0.0,27.0,0.0,25.0,79.0,0.0,20.0,174.0,26.0,...,0.0,61.0,0.0,61.0,4.0,0.0,61.0,4.0,0.0,0.0
2002-01-01 00:35:00,0.0,0.0,27.0,0.0,29.0,84.0,0.0,12.0,152.0,26.0,...,0.0,62.0,0.0,62.0,4.0,0.0,62.0,4.0,0.0,0.0
2002-01-01 00:40:00,0.0,0.0,35.0,0.0,25.0,82.0,0.0,18.0,179.0,23.0,...,0.0,60.0,0.0,60.0,4.0,0.0,60.0,4.0,0.0,0.0
